# Assignment 11.1 — Demand Prediction and Pricing Optimization with Gurobi

Building a **demand prediction model** and a **pricing optimization model** for an amusement park souvenir, using:

- A **linear additive demand model** with price, lagged prices, and seasonality  
- A **discrete price ladder** as a business rule  
- A **revenue‑maximizing objective** over Weeks 157–164  

We need to extract and plot the **optimal weekly prices**.

In [ ]:
# Install and import Gurobi and Plotly packages

!pip install gurobipy plotly --quiet

In [1]:
# Import the packages 
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import plotly.express as px

## 1. Planning Horizon and Indexing

We consider Weeks 157–164 as the planning horizon.  
We will model all eight weeks, but per the assignment, Week 164 does not need to appear in the final output table of predicted prices.

In [2]:
# Define planning horizon: weeks 157–164
weeks = list(range(157, 165))
print("Weeks (t):", weeks)

# Convenience aliases for the first two weeks
w1 = weeks[0]
w2 = weeks[1]
w1, w2

Weeks (t): [157, 158, 159, 160, 161, 162, 163, 164]


(157, 158)

## 2. Demand Model Parameters

The demand model is a **linear additive regression**:

$$
d_t = 1.978242858 
- 2.809634145\,p_t 
+ 0.963410728\,p_{t-1} 
+ 0.759639170\,p_{t-2}
+ \sum_{s=2}^{13} \beta_s \cdot \text{Season}_s
$$

This captures:

- Own‑price effect (negative coefficient on $(p_t)$)  
- Lagged price effects ($(p_{t-1}, p_{t-2})$)  
- Seasonal variation via dummy variables $(\text{Season}_2, \dots, \text{Season}_{13})$.

In [3]:
# Demand model parameters (from assignment)
intercept = 1.978242858

p_coeff  = -2.809634145   # coefficient on p_t
p1_coeff =  0.963410728   # coefficient on p_{t-1}
p2_coeff =  0.759639170   # coefficient on p_{t-2}

# Seasonal coefficients: Season1 is baseline (0), Seasons 2–13 have explicit coefficients
season_coeff = {
    1: 0.0,
    2: -0.562046910,
    3:  0.087545274,
    4: -0.402637480,
    5: -0.027326010,
    6:  0.004349885,
    7: -0.036102297,
    8: -0.069280527,
    9:  0.160276197,
    10: 1.104208897,
    11: 1.122711287,
    12: 1.176802194,
    13: 0.947945548
}

## 3. Season Mapping

The year is divided into **13 seasons**, each consisting of 4 weeks.  
We map each week in the planning horizon to a season index $(s \in \{1,\dots,13\})$.

In [4]:
# Map each week to a season (1–13), assuming 52 weeks/year and 4 weeks/season
season = {}
for w in weeks:
    week_in_year = ((w - 1) % 52) + 1  # wrap into 1–52
    s = int(np.ceil(week_in_year / 4)) # 4 weeks per season
    season[w] = s

season

{157: 1, 158: 1, 159: 1, 160: 1, 161: 2, 162: 2, 163: 2, 164: 2}

## 4. Price Ladder (Business Rule)

Prices must be chosen from the following discrete **price ladder**:

- 1.00  
- 0.95  
- 0.85  
- 0.75  
- 0.60  
- 0.50  

We will enforce this using **binary decision variables** that select exactly one ladder price per week.

In [5]:
# Six-level price ladder
price_ladder = [1.00, 0.95, 0.85, 0.75, 0.60, 0.50]
price_ladder

[1.0, 0.95, 0.85, 0.75, 0.6, 0.5]

## 5. Optimization Model with Gurobi

We now construct a **mixed‑integer quadratic program**:

- **Decision variables**
  - Continuous: $(p_t)$ (price in week t)
  - Binary: $(x_{t,k})$ (1 if ladder price k is chosen in week t, 0 otherwise)

- **Objective function**
  - Maximize total revenue: $(\sum_t p_t \cdot d_t)$

- **Constraints**
  - Price ladder selection: exactly one ladder price per week
  - Price definition: $(p_t = \sum_k k \cdot x_{t,k})$
  - Demand defined by the linear additive model

In [6]:
# Create Gurobi model
mod = gp.Model("amusement_souvenir_pricing")

# Decision variables
p = mod.addVars(weeks, name="price")                          # continuous prices
x = mod.addVars(weeks, price_ladder, vtype=GRB.BINARY, name="x")  # ladder selection

Restricted license - for non-production use only - expires 2027-11-29


### 5.1 Demand Expression

We need not create explicit demand variables; instead, we can probably embed the demand formula directly into the **objective function** as a quadratic expression in $(p_t)$ and lagged prices.

In [8]:
# Helper Method: demand expression for a given week t
def demand_expr(t):
    """Return a Gurobi LinExpr for d_t as a function of p_t, p_{t-1}, p_{t-2}, and season."""
    s = season[t]
    base = intercept + season_coeff[s]
    
    # For the first two weeks, we need initial lagged prices.
    # A simple assumption: use baseline price 1.0 for missing lags.
    if t == w1:
        p_t   = p[t]
        p_tm1 = 1.0
        p_tm2 = 1.0
    elif t == w2:
        p_t   = p[t]
        p_tm1 = p[w1]
        p_tm2 = 1.0
    else:
        p_t   = p[t]
        p_tm1 = p[t-1]
        p_tm2 = p[t-2]
    
    return base + p_coeff*p_t + p1_coeff*p_tm1 + p2_coeff*p_tm2

### 5.2 Objective Function

We need to maximize total revenue over Weeks 157–164:

$$
\max \sum_{t=157}^{164} p_t \cdot d_t
$$

where $(d_t)$ is given by the demand expression above.

In [9]:
# Set objective: maximize sum_t p_t * d_t
revenue_expr = gp.quicksum(p[t] * demand_expr(t) for t in weeks)
mod.setObjective(revenue_expr, GRB.MAXIMIZE)

## Adding contraints
### 5.3 Price Ladder Constraints

1. **Exactly one ladder price per week**  
   $$
   \sum_{k \in \text{ladder}} x_{t,k} = 1 \quad \forall t
   $$

2. **Price definition**  
   $$
   p_t = \sum_{k \in \text{ladder}} k \cdot x_{t,k} \quad \forall t
   $$

In [10]:
# Exactly one ladder price per week
mod.addConstrs(
    (gp.quicksum(x[t, k] for k in price_ladder) == 1 for t in weeks),
    name="select_one_price"
)

# Price equals selected ladder value
mod.addConstrs(
    (p[t] == gp.quicksum(k * x[t, k] for k in price_ladder) for t in weeks),
    name="price_definition"
)

{157: <gurobi.Constr *Awaiting Model Update*>,
 158: <gurobi.Constr *Awaiting Model Update*>,
 159: <gurobi.Constr *Awaiting Model Update*>,
 160: <gurobi.Constr *Awaiting Model Update*>,
 161: <gurobi.Constr *Awaiting Model Update*>,
 162: <gurobi.Constr *Awaiting Model Update*>,
 163: <gurobi.Constr *Awaiting Model Update*>,
 164: <gurobi.Constr *Awaiting Model Update*>}

## 6. Solve the Model

Calling Gurobi’s optimizer to solve the mixed‑integer quadratic program and obtain the optimal weekly prices.

In [11]:
mod.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i7-6700K CPU @ 4.00GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 56 columns and 104 nonzeros (Max)
Model fingerprint: 0xfbc9a1f9
Model has 8 linear objective coefficients
Model has 21 quadratic objective terms
Variable types: 8 continuous, 48 integer (48 binary)
Coefficient statistics:
  Matrix range     [5e-01, 1e+00]
  Objective range  [1e+00, 4e+00]
  QObjective range [2e+00, 6e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 5.0955328
Presolve removed 0 rows and 8 columns
Presolve time: 0.01s
Presolved: 16 rows, 48 columns, 88 nonzeros
Presolved model has 21 quadratic objective terms
Variable types: 8 continuous, 40 integer (40 binary)

Root relaxation: objective 6.242180e+00, 21 iterations, 0.00 seconds (0.00 work

## 7. Extract and Display Optimal Prices

We extract the optimal prices for Weeks 157–164, but per the assignment, we can focus the **reported output** on Weeks 157–163.

In [12]:
# Collect optimal prices into a DataFrame
df_prices = pd.DataFrame(index=weeks, columns=["optimal_price"])
for t in weeks:
    df_prices.loc[t, "optimal_price"] = p[t].X

df_prices

,optimal_price
157,0.95
158,0.95
159,0.85
160,0.85
161,0.75
162,0.6
163,0.5
164,0.5


In [13]:
# Restrict to Weeks 157–163 for reporting
df_report = df_prices.loc[157:163].copy()
df_report

,optimal_price
157,0.95
158,0.95
159,0.85
160,0.85
161,0.75
162,0.6
163,0.5
